# WaRS — huấn luyện trên Kaggle

**Chuẩn bị (một lần):**
1. *Add Input* → dataset **MaSTr1325** (ảnh + mask) và dataset **USV-VN** (`usv_vn.zip` có `train/valid/test`), upload dạng private.
2. *Settings* (panel phải): Accelerator = **GPU T4 x2** hoặc **P100**; Internet = **On** (cần xác minh số điện thoại).

**Chạy:** sửa ô cấu hình bên dưới → **Save Version → Save & Run All (Commit)**. Notebook chạy nền tối đa 12 giờ; kết quả nằm ở tab *Output* của version.

**Chạy tiếp lần sau:** *Add Input* → *Your Work* → chọn output của version trước; ô "chạy tiếp" sẽ tự chép `last.pt`.

In [ ]:
# ===== Cấu hình =====
GITHUB_USER = "huyhau0504-cmd"
REPO = "WaRS"
BRANCH = "main"
RUN = "ewasr_h"        # tên file trong configs/ (ewasr_h, ewasr, ...)
TRAIN_SEG = True       # huấn luyện eWaSR / eWaSR-H trên MaSTr1325
TRAIN_YOLO = False     # huấn luyện YOLOv8n trên USV-VN
EXTRA_SET = ""         # override thêm, vd. "train.epochs=50 data.vn_color_aug=true"
TIME_LIMIT_H = 11      # dừng an toàn trước giới hạn 12 giờ của Kaggle

In [ ]:
import os, subprocess
from pathlib import Path
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                f"https://github.com/{GITHUB_USER}/{REPO}.git", "/kaggle/working/WaRS"], check=True)
%cd /kaggle/working/WaRS
!git log -1 --oneline
!pip install -q albumentations onnx onnxruntime onnxsim
!mkdir -p weights && curl -sL -o weights/ewasr_resnet18.pth https://github.com/tersekmatija/eWaSR/releases/download/0.1.0/ewasr_resnet18.pth

In [ ]:
# Tự tìm dữ liệu trong /kaggle/input
def find(pattern):
    return next(Path("/kaggle/input").rglob(pattern), None)

usv_img = find("left_*.jpg")
usv_root = usv_img.parents[2] if usv_img else None          # .../usv_vn
mask = find("*m.png")
if mask:
    mask_dir = mask.parent
    image_dir = next(p.parent for p in Path("/kaggle/input").rglob(mask.name[:-5] + ".jpg"))
    mastr_root = Path(os.path.commonpath([mask_dir, image_dir]))
print("USV-VN:", usv_root)
print("MaSTr1325:", (mastr_root, image_dir, mask_dir) if mask else None)
assert usv_root, "Chưa Add Input dataset USV-VN"
assert mask or not TRAIN_SEG, "Chưa Add Input dataset MaSTr1325"

In [ ]:
# Chạy tiếp từ output của version trước (nếu đã Add Input)
OUT = f"/kaggle/working/runs/{RUN}"
prev = find(f"runs/{RUN}/last.pt")
if prev and TRAIN_SEG:
    os.makedirs(OUT, exist_ok=True)
    !cp -n {prev.parent}/* {OUT}/
    print("Tiếp tục từ", prev)

In [ ]:
if TRAIN_SEG:
    !python tools/train.py --config configs/{RUN}.yaml --out {OUT} --time-limit-h {TIME_LIMIT_H} \
        --set data.mastr_root={mastr_root} data.image_dir={image_dir.relative_to(mastr_root)} \
              data.mask_dir={mask_dir.relative_to(mastr_root)} train.workers=4 {EXTRA_SET}

In [ ]:
# Xuất ONNX + đánh giá nhanh trên USV-VN (chỉ Recall nếu chưa có data/usv_vn/water_edges.json trong repo)
if TRAIN_SEG and Path(f"{OUT}/best.pth").exists():
    !python tools/export_onnx.py --model {RUN} --weights {OUT}/best.pth --size 384 512 --out {OUT}/{RUN}_384x512.onnx
    !python tools/predict.py --model {RUN} --weights {OUT}/best.pth --usv-root {usv_root} --split test --out {OUT}/pred_usv_test
    !python tools/evaluate_usv.py --pred-masks {OUT}/pred_usv_test/masks --usv-root {usv_root} --split test --size-edges 56 70 --out-json {OUT}/usv_test.json

## YOLOv8n trên USV-VN (mô hình đối chứng)

In [ ]:
if TRAIN_YOLO:
    !pip install -q ultralytics
    import yaml
    cfg = yaml.safe_load(open("configs/usv_vn_yolo.yaml")); cfg["path"] = str(usv_root)
    yaml.safe_dump(cfg, open("/kaggle/working/usv_vn_yolo.yaml", "w"))
    Y = "/kaggle/working/runs/yolo"
    !yolo detect train data=/kaggle/working/usv_vn_yolo.yaml model=yolov8n.pt imgsz=640 epochs=100 project={Y} name=usv_vn exist_ok=True
    !python tools/yolo_predict.py --weights {Y}/usv_vn/weights/best.pt --usv-root {usv_root} --split test --out {Y}/pred_usv_test
    !python tools/evaluate_usv.py --pred-boxes {Y}/pred_usv_test/boxes.json --usv-root {usv_root} --split test --size-edges 56 70 --out-json {Y}/usv_test.json
    # ONNX cho Hailo Model Zoo (hailomz compile yolov8n --ckpt ... --hw-arch hailo8l --classes 1)
    !yolo export model={Y}/usv_vn/weights/best.pt format=onnx opset=11 imgsz=640

In [ ]:
# Dọn bản clone để tab Output chỉ còn kết quả
%cd /kaggle/working
!rm -rf /kaggle/working/WaRS